# Simulação do Algoritmo de Criptografia Assimétrica Pós-Quântica (PQC) CRYSTALS-Kyber

Nessa simulação pedagógica iremos ver passo a passo sobre como conceitualmente o Kyber funciona, no qual é um mecanismo de encapsulamento de chaves (KEM) de criptografia assimétrica pós-quântica projetado para o estabelecimento seguro de chaves compartilhadas em protocolos de comunicação. Ele pertence à família de algoritmos baseados em **Module-LWE (Learning With Errors)**. Diferentemente do LWE tradicional, que trabalha com vetores, o Module-LWE opera sobre módulos de polinômios, permitindo um equilíbrio entre segurança, desempenho e tamanho das chaves. Os parâmetros usados são reduzidos propositalmente para entedermos como é o processo do algoritmo passo a passo e podermos visualizar melhor o mesmo, aqui usamos vetores com um vetor de erro adicionado para simular o Module-LWE.



In [ ]:
# Importação do numpy para operações com matrizes e vetores
# e do hashlib para hashes.
import numpy as np
import hashlib

### Legenda de Parâmetros e Variáveis Globais

- **n_coeficientes** = **n** ou **n_coeficientes**, equivale a quantidade de coeficientes no Kyber real (256). Aqui vai ser o tamanho máximo dos vetores a serem utilizados.
- **q** = **q** é a variável que guarda o valor modular a ser comparado, por exemplo, **mod (q)**, que equivalente ao restante de uma operação de divisão no qual há alguma sobra, como por exemplo x // q = y, no python a operação de modulo é o '**%**'. Isso ajuda a limitar os valores para dentro de uma lista de valores entre 0 a q-1, evitando assim ter valores muito altos. No Kyber o parâmetro **q** é 3329. garantido que os valores fiquem entre 0 a 3328. Nessa simulação iremos usar o q como igual a 17.
- **k** = o parâmetro **k** indica a quantidade de polinômios usados no algoritmo. Aqui estamos usando k = 2 para indicar que vamos usar dois vetores (ou polinômios no Kyber real).
- **eta** = o parâmetro **eta** no Kyber é utilizado para gerar erros com coeficientes muito pequenos dentro de um alcance '**eta**'. Aqui vamos utilizar o eta como 1, onde o vetor de erro vai variar entre -1 a 1. Que será o nível de ruído implementado à operação.

In [ ]:
# Definição dos parâmetros globais a serem utilizados.
n_coeficientes = 8
q = 17
k = 2
eta = 1

In [ ]:
# Função que gera um vetor de valores randômicos entre -eta e eta.
def sample_eta(size, eta):
    return np.random.randint(-eta,eta + 1,size=size)

### KeyGen (Geração de par de chaves sk e pk)

Nessa etapa iremos simular o algoritmo de geração de chaves (KeyGen) do Kyber.

In [ ]:
# Aqui criamos o nosso vetor de segredo ou s de k = 2 de acordo ao valor
# do parâmetro eta, no qual fará parte da chave privada sk.
secret = np.array([sample_eta(n_coeficientes, eta) for _ in range(k)])

secret, secret.shape

(array([[ 0, -1,  1,  1, -1, -1, -1,  0],
        [-1, -1,  1,  0,  0,  0, -1, -1]]),
 (2, 8))

In [ ]:
# Vetor de erro (ou ruído) a ser adicionado ao cálculo da chave pública pk.
error = np.array([sample_eta(n_coeficientes, eta) for _ in range(k)])

error, error.shape

(array([[ 1,  0, -1,  0,  1, -1,  0, -1],
        [-1,  1, -1,  1, -1, -1, -1,  0]]),
 (2, 8))

O vetor de erro é o elemento fundamental da segurança baseada em Learning With Errors (LWE). Sem esse pequeno ruído, a recuperação da chave privada a partir da chave pública se reduziria a um sistema linear convencional, que poderia ser resolvido de forma eficiente. O ruído torna esse problema computacionalmente difícil, formando a base matemática da segurança do Kyber.

In [ ]:
# Geração da matriz aleatória A de tamanho k x k.
A = np.random.randint(0, q, size=(k, k))

A, A.shape

(array([[ 5,  2],
        [ 6, 11]]),
 (2, 2))

## Fórmula de geração da chave pública '**t**'

O Kyber real usa como base para a criação da chave pública 't' a fórmula

$t = A * s + e \ (mod \ q)$

no qual os valores são normalizados a 'q' dentro de um anel de polinômios

$R_q=ℤ_q[x]/(x^n+1)$

no qual garante que os polinômios que existem em A e s não ultrapassem os valores definidos em q = 3329 e n = 256 no algoritmo real.

Como não estamos utilizando polinômios nessa simulação, iremos apenas utilizar a fórmula base reduzido a 'q' para melhor entendimento.



In [ ]:
# Criação da chave pública de acordo a fórmula base do Kyber,
# note que a multiplicação de matrizes no numpy é feita pelo
# operador '@'.
t = (A @ secret + error) % q

t, t.shape

(array([[16, 10,  6,  5, 13, 11, 10, 14],
        [ 5,  1, 16,  7, 10, 10, 16,  6]]),
 (2, 8))

In [ ]:
# E no final da etapa de KeyGen do Kyber temos um 'sk', secret_key,
# e um 'pk', public_key, no qual só podemos compartilhar o 'pk'.
sk = (secret[0], secret[1])
pk = (t[0], t[1], A)

print(f"Chave privada: {sk}")
print(f"Chave pública: {pk}")

Chave privada: (array([ 0, -1,  1,  1, -1, -1, -1,  0]), array([-1, -1,  1,  0,  0,  0, -1, -1]))
Chave pública: (array([16, 10,  6,  5, 13, 11, 10, 14]), array([ 5,  1, 16,  7, 10, 10, 16,  6]), array([[ 5,  2],
       [ 6, 11]]))


In [ ]:
# Para fins de validação, podemos verificar passando as nossas chaves sk e pk
# na fórmula novamente e conferir se teremos o mesmo 't'.
t_check = (pk[2] @ np.array([sk[0], sk[1]]) + error) % q

print(f"t gerado a partir do sk e pk: {t_check}")
print(f"t_check é igual a t? {"Sim" if np.array_equal(t, t_check) else "Não"} ")

t gerado a partir do sk e pk: [[16 10  6  5 13 11 10 14]
 [ 5  1 16  7 10 10 16  6]]
t_check é igual a t? Sim 


## Encapsulamento

O Kyber utiliza o modelo **KEM (Key Encapsulation Mechanism)** para gerar uma **chave compartilhada (shared_key ou K)** para ser usada no compartilhamento da mesma entre as partes, e nessa etapa iremos simular o algoritmo de geração de um shared_key e encapsulação usando um byte aleatório como parâmetro mu.

In [ ]:
# Função que gera um vetor de valores randômicos entre -eta e eta
# com uma seed específica.
def sample_eta_encaps(seed, n, eta):

    stream = hashlib.shake_256(seed).digest(n)

    return np.array([
        (b % (2*eta+1)) - eta
        for b in stream
    ])

O SHAKE-256 pertence à família SHA-3 e possui saída de tamanho variável (XOF - eXtendable Output Function). Essa característica permite gerar quantidades arbitrárias de bytes pseudoaleatórios a partir de uma única seed, sendo ideal para derivar vetores de erro e segredos temporários.

In [ ]:
# Gerando o byte aleatório 'mu' para o encapsulamento.
mu = np.random.bytes(1)

mu

b'i'

In [ ]:
# Converte os bytes em um array numpy de inteiros para fins de cálculo
mu_array = np.frombuffer(mu, dtype=np.uint8)

mu_array

array([105], dtype=uint8)

In [ ]:
# Desse array de inteiros convertido ele expande em forma de um array de 8 bits
# para ser compatível com a simulação atual.
bits = np.unpackbits(mu_array)

bits, bits.shape

(array([0, 1, 1, 0, 1, 0, 0, 1], dtype=uint8), (8,))

In [ ]:
# Agora temos que converte os bits 0 e 1 para 0 ou 8, para ser distinguível
# na hora da decodificação
mu_vec = bits * (q // 2)

mu_vec, mu_vec.shape

(array([0, 8, 8, 0, 8, 0, 0, 8], dtype=uint8), (8,))

No Kyber real é utilizado um parâmetro 'r' que é derivado através de uma função G da compressão da chave pk em bytes e do vetor mu, gerando um hash de 64 bytes no qual é dividido igualemnte em uma chave intermediária K_bar e no parâmetro 'r' que é usado para gerar um fluxo de bytes aleatórios para a geração do cipher text 'c'.

In [ ]:
# Aqui primeiro fazemos uma compressão simples concatenando os bytes da chave pk
pk_compress = pk[0].tobytes() + pk[1].tobytes() + pk[2].tobytes()

# e fazemos o hash da mesma
hash_pk = hashlib.sha3_256(pk_compress).digest()

# após o hash(pk), temos que fazer o hash de mu + h(pk) que é a função G
hash_input = mu + hash_pk

# um ponto importante aqui é que ao fazer a função G, temos que gerar o hash
# usando o sha3_512 e não o 256, pois o mesmo retorna um array de 64 bytes
# para podermos fazer a divisão em K-bar e r cada uma com 32 bytes.
# O sha3_256 só retorna um array de 32 bytes.
G = hashlib.sha3_512(hash_input).digest()

# e aqui temos nossa chave intermediária K_bar e o nosso 'r' que será usado para
# gerar os seed de randomização.
K_bar = G[:32]
r = G[32:]

K_bar, r

(b'\xb5C\x83\xc9\x01=\xc1\x1e\r\x05F\x12\x8dp\xc6\x91\xdf.\xf9\x92A\xaf\\b\xca\x87\xd4[Q\xb2e\xf1',
 b'\xfc@\x9c\x15\xec+\xa8\xdc\xe0X\x91q\xf9\xba\xcbx\x1f~{\x9c\xfc\xdb\xc2\xd0o\xb7\xd2\xfd\xeeS\\\xa9')

Embora K̄ seja derivada durante o encapsulamento, ela não é utilizada diretamente como chave compartilhada. O Kyber realiza um último hash utilizando K̄ e H(c), no qual chamamos de KDF (Key Derivation Function), produzindo a shared key final K. Esse procedimento vincula a chave ao ciphertext e fortalece a segurança contra modificações do encapsulamento.

In [ ]:
# Criamos aqui uma função pseudo-randômica (PRF) para ser usada no fluxo de geração
# de bytes aleatórios para gerar 'c'
def prf(seed, nonce, outlen):
    shake = hashlib.shake_256()
    shake.update(seed)
    shake.update(bytes([nonce]))
    return shake.digest(outlen)

In [ ]:
# Usando a função PRF, criamos 5 seeds para usar em seus respectivos vetores no
# encapsulamento de 'c'.
seed_rs0 = prf(r, 0, 64)
seed_rs1 = prf(r, 1, 64)

seed_err_u0 = prf(r, 2, 64)
seed_err_u1 = prf(r, 3, 64)

seed_err_v = prf(r, 4, 64)

seed_rs0, seed_rs1, seed_err_u0, seed_err_u1, seed_err_v

(b'\xf7(\x07C\xdc\x91`\x02\xdc\xe3`\xdbf\x8a\x113QQ\xec%\x1e\x01.n\xc4\xceTX\xeb\xee\x7f\xb9\xab2\x96\xf4\xae\xc9\x9f\\o?\xbf\xc7\xb5\xc5\xc2\x98N&q5/XY\x1c\xfc6\xcd\x84\xf1\xcan\x12',
 b"qb=*\\A?\xd5\x0b)^\xf0'\x00e\xe2c\xb4/\x88\x87\x03\x06\xb7U\xec\xbcn\xff\xeahq\x93\xa3\xbc\x88.V\x85lb\xd2]\x1a\xc6e^\xd1!q\xecz\x0e\xb0'KE\xc4n\xb2\xd6\xfcTu",
 b'\xaa\x08\x0flH\x89\xd7\xe1\x92cm^a-zD\xa5"\xb6\x07#\xe9\xe8\x9c@\x0ed\xbaiQ,\xe1]\xd2(s\xf0\x1f\xbf\xe2WO\xe5\xf6\x06\x1cE\xc8\xef\x90\\R\x8d\xefV\xbc\x07,\n\x88\xe1\x19\\\xaa',
 b'\xdb]\x87\x1d\xce\xab\x96\xb9\x87*\xfe\x0e\x13\x8d\x1e\x11Q\xc2~\xec\x0c\xf3\x02\xe2\xe8\xc8.0Ex>UT\x03\x90)M\x82q\xad\x10I\xa7\xf5\x98y\x89~\x0f\x96,\xb7\xea\xfc\x13\x95\x98\xed\xe5\xee\xd9[\xda\xc8',
 b'\x96~\xfd\xdb\x84o\x0b9\xe9C\xe5\xdc\xfbq\x01\x1c#\xd6\x0f\x11J7\x95m\xf7Fen\x11\xc0n\x86\xc9\xd7\xcd\xc6\x88|\xa87x^\xa2\t\x87\x8c^\xc3Z\xf8zV\x14\x86\xe0\\}\xa9~\xad\xeb\xc3\xd6\x98')

In [ ]:
# Temos que criar um vetor aleatório com base em eta e o seed provido de 'r'
# para ser usado como vetor de segredo (ou s) temporário
random_secret = np.array([
    sample_eta_encaps(seed_rs0, n_coeficientes, eta),
    sample_eta_encaps(seed_rs1, n_coeficientes, eta)
])

random_secret, random_secret.shape

(array([[ 0,  1,  1,  0,  0, -1,  1,  1],
        [ 0,  1, -1, -1, -1,  1,  1, -1]]),
 (2, 8))

In [ ]:
# Agora criamos 2 vetores de erro (e1.shape -> (2, 8), e2.shape -> (8,))
# com base em eta para codificar o vetor u do cipher text e o vetor v do mesmo.
error_u = np.array([
    sample_eta_encaps(seed_err_u0, n_coeficientes, eta),
    sample_eta_encaps(seed_err_u1, n_coeficientes, eta)])

error_v = sample_eta_encaps(seed_err_v, n_coeficientes, eta)

error_u, error_u.shape, error_v, error_v.shape

(array([[-1,  1, -1,  0,  1,  0, -1,  0],
        [ 1,  0,  1,  1,  0,  1,  1, -1]]),
 (2, 8),
 array([-1,  1,  1,  1,  0,  1,  1,  1]),
 (8,))

### Codificação em 'u' e 'v'

O Kyber gera dois vetores (u, v) para o cipher text no qual o vetor 'u' é gerado com base na fórmula

$u=A^Tr+e \ (mod \ q)$

onde 'r' e 'e' aqui são o **random_secret** e **error_u** no nosso caso. E para o vetor 'v' temos a fórmula

$v = t^Tr + e + m \ (mod \ q)$

onde temos o produto interno de 't' (nossa chave pública disposta em pk) e 'r' (random_secret nesse caso), e 'm' é o vetor mu a ser codificado.

In [ ]:
# Geração do vetor u. Obs.: pk[2] é a matriz A
u = (pk[2].T @ random_secret + error_u) % q

u, u.shape

(array([[16, 12, 15, 11, 12,  1, 10, 16],
        [ 1, 13,  9,  7,  6, 10, 14,  7]]),
 (2, 8))

In [ ]:
# Geração do vetor v. Obs.: pk[0] e pk[1] são t[0] e t[1] respectivamente
t_pk = np.array([pk[0], pk[1]])

v = (np.sum(t_pk * random_secret, axis=0) + error_v + mu_vec) % q

v, v.shape

(array([16,  3, 16, 11, 15,  0, 10,  0]), (8,))

In [ ]:
# Aqui guardamos o cipher text que será compartilhado entre as partes
cipher_txt = (u, v)

cipher_txt

(array([[16, 12, 15, 11, 12,  1, 10, 16],
        [ 1, 13,  9,  7,  6, 10, 14,  7]]),
 array([16,  3, 16, 11, 15,  0, 10,  0]))

In [ ]:
# Aqui fazemos uma pequena compressão dos bytes do cypher text
# para termos o hash(c) do mesmo.
c = cipher_txt[0].tobytes() + cipher_txt[1].tobytes()

hash_c = hashlib.sha3_256(c).digest()

In [ ]:
# Tendo hash(c) conseguimos criar nossa shared_key (K) fazendo o hash da chave
# intermediária K_bar e hash(c).
K = hashlib.sha3_256(K_bar + hash_c).hexdigest()

K

'cbdec1d9fd14d8d34693ec327083904d1b2c6716d1fbb6852994484d97a47cbe'

## Desencapsulação

Tendo o cipher text, podemos agora o desencapsular para obter K do mesmo através da nossa chave privada sk. Primeiro executamos a seguinte fórmula:

$d=s^Tu \ (mod \ q)$

para obtermos de volta o mu com:

$m=v-d \ (mod \ q)$

atráves do cipher text, e com esse mu temos que fazer a re-encapsulação do mesmo para gerar novamente o mesmo cipher text e obtermos o K validado.

A reencapsulação é responsável por verificar se o ciphertext recebido poderia realmente ter sido produzido pela chave pública correspondente. Caso qualquer byte tenha sido alterado, o ciphertext reconstruído será diferente, impedindo a geração da mesma chave compartilhada.

In [ ]:
# Primeira parte da densencapsulação do cipher text
sk = np.array([secret[0], secret[1]])

d = np.sum(sk * cipher_txt[0], axis=0) % q

d, d.shape

(array([16,  9,  7, 11,  5, 16, 10, 10]), (8,))

In [ ]:
# Primeira parte da densencapsulação do cipher text com o mu recuperado
mu_d = (cipher_txt[1] - d) % q

mu_d, mu_d.shape

(array([ 0, 11,  9,  0, 10,  1,  0,  7]), (8,))

In [ ]:
# Após recuperar o nosso mu, precisar fazer um threshold nele para
# termos os bits originais antes de termos feito a encapsulação,
# fazendo a distinção do vetor recuperado para definir o que está mais
# perto de 0 ou mais perto de 8 como mencionamos na parte da encapsulação
mu_centered = np.where(
    mu_d > q//2,
    mu_d - q,
    mu_d
)

bits_rec = (
    np.abs(mu_centered)
    >
    q//4
).astype(np.uint8)

# E por fim nessa etapa temos nosso mu recuperado em bytes
mu_rec = np.packbits(bits_rec)

mu_rec.tobytes()

b'i'

In [ ]:
# Note que o mu_rec deve ser o mesmo que o mu antes da encapsulação
mu_rec.tobytes() == mu

True

## Re-Encapsulação

Agora precisamos re-encapsular o mu recuperado com o nosso pk novamente para termos no final o K validado.

In [ ]:
# Comprimindo e fazendo hash(pk) novamente e passando na função G para termos o
# K_bar desse mu recuperado e o parâmetro para ramdomização 'r' do mesmo
# para refazermos o cipher text que será o c'.
hash_pk_rec = hashlib.sha3_256(pk[0].tobytes() + pk[1].tobytes() + pk[2].tobytes()).digest()

G_rec = hashlib.sha3_512(mu_rec.tobytes() + hash_pk_rec).digest()

K_bar_rec = G_rec[:32]
r_rec = G_rec[32:]

K_bar_rec, r_rec

(b'\xb5C\x83\xc9\x01=\xc1\x1e\r\x05F\x12\x8dp\xc6\x91\xdf.\xf9\x92A\xaf\\b\xca\x87\xd4[Q\xb2e\xf1',
 b'\xfc@\x9c\x15\xec+\xa8\xdc\xe0X\x91q\xf9\xba\xcbx\x1f~{\x9c\xfc\xdb\xc2\xd0o\xb7\xd2\xfd\xeeS\\\xa9')

In [ ]:
# E da mesma forma que fizemos na encapsulação, temos que fazer o fluxo de bytes
# aleatórios com base no r vindo do cipher text recuperado, no qual todos os
# seeds devem ser iguais ao usado na primeira encapsulação.
seed_rs0_check = prf(r_rec, 0, 64)
seed_rs1_check = prf(r_rec, 1, 64)

seed_err_u0_check = prf(r_rec, 2, 64)
seed_err_u1_check = prf(r_rec, 3, 64)

seed_err_v_check = prf(r_rec, 4, 64)

seed_rs0_check, seed_rs1_check, seed_err_u0_check, seed_err_u1_check, seed_err_v_check

(b'\xf7(\x07C\xdc\x91`\x02\xdc\xe3`\xdbf\x8a\x113QQ\xec%\x1e\x01.n\xc4\xceTX\xeb\xee\x7f\xb9\xab2\x96\xf4\xae\xc9\x9f\\o?\xbf\xc7\xb5\xc5\xc2\x98N&q5/XY\x1c\xfc6\xcd\x84\xf1\xcan\x12',
 b"qb=*\\A?\xd5\x0b)^\xf0'\x00e\xe2c\xb4/\x88\x87\x03\x06\xb7U\xec\xbcn\xff\xeahq\x93\xa3\xbc\x88.V\x85lb\xd2]\x1a\xc6e^\xd1!q\xecz\x0e\xb0'KE\xc4n\xb2\xd6\xfcTu",
 b'\xaa\x08\x0flH\x89\xd7\xe1\x92cm^a-zD\xa5"\xb6\x07#\xe9\xe8\x9c@\x0ed\xbaiQ,\xe1]\xd2(s\xf0\x1f\xbf\xe2WO\xe5\xf6\x06\x1cE\xc8\xef\x90\\R\x8d\xefV\xbc\x07,\n\x88\xe1\x19\\\xaa',
 b'\xdb]\x87\x1d\xce\xab\x96\xb9\x87*\xfe\x0e\x13\x8d\x1e\x11Q\xc2~\xec\x0c\xf3\x02\xe2\xe8\xc8.0Ex>UT\x03\x90)M\x82q\xad\x10I\xa7\xf5\x98y\x89~\x0f\x96,\xb7\xea\xfc\x13\x95\x98\xed\xe5\xee\xd9[\xda\xc8',
 b'\x96~\xfd\xdb\x84o\x0b9\xe9C\xe5\xdc\xfbq\x01\x1c#\xd6\x0f\x11J7\x95m\xf7Fen\x11\xc0n\x86\xc9\xd7\xcd\xc6\x88|\xa87x^\xa2\t\x87\x8c^\xc3Z\xf8zV\x14\x86\xe0\\}\xa9~\xad\xeb\xc3\xd6\x98')

Note que até então que o resultado das duas primeiras células dessa etapa tem o mesmo resultado que na etapa de encapsulação.

In [ ]:
# Regerando o vetor random_secret com o r recuperado.
random_secret_check = np.array([
    sample_eta_encaps(seed_rs0_check, n_coeficientes, eta),
    sample_eta_encaps(seed_rs1_check, n_coeficientes, eta)
])

random_secret_check, random_secret_check.shape

(array([[ 0,  1,  1,  0,  0, -1,  1,  1],
        [ 0,  1, -1, -1, -1,  1,  1, -1]]),
 (2, 8))

In [ ]:
# E também regerando os vetores de erro (ruído).
error_u_check = np.array([
    sample_eta_encaps(seed_err_u0_check, n_coeficientes, eta),
    sample_eta_encaps(seed_err_u1_check, n_coeficientes, eta)])

error_v_check = sample_eta_encaps(seed_err_v_check, n_coeficientes, eta)

error_u_check, error_u_check.shape, error_v_check, error_v_check.shape

(array([[-1,  1, -1,  0,  1,  0, -1,  0],
        [ 1,  0,  1,  1,  0,  1,  1, -1]]),
 (2, 8),
 array([-1,  1,  1,  1,  0,  1,  1,  1]),
 (8,))

In [ ]:
# Gerando o vetor u do novo cipher text.
u_check = (pk[2].T @ random_secret_check + error_u_check) % q

u_check, u_check.shape

(array([[16, 12, 15, 11, 12,  1, 10, 16],
        [ 1, 13,  9,  7,  6, 10, 14,  7]]),
 (2, 8))

In [ ]:
# Do mu temos que descompactar os bits de novo
bits_rec = np.unpackbits(mu_rec)

mu_vec_rec = bits * (q // 2)

mu_vec_rec, mu_vec_rec.shape

(array([0, 8, 8, 0, 8, 0, 0, 8], dtype=uint8), (8,))

In [ ]:
# Validamos a igualdade dos vetores mu
mu_vec, mu_vec_rec

(array([0, 8, 8, 0, 8, 0, 0, 8], dtype=uint8),
 array([0, 8, 8, 0, 8, 0, 0, 8], dtype=uint8))

In [ ]:
# E fazemos o vetor v com o vetor mu recuperado descompactado.
v_check = (np.sum(t_pk * random_secret_check, axis=0) + error_v_check + mu_vec_rec) % q

v_check, v_check.shape

(array([16,  3, 16, 11, 15,  0, 10,  0]), (8,))

In [ ]:
# Agora temos o cipher text recriado com base no mu recuperado.
cipher_txt_check = (u_check, v_check)

cipher_txt_check

(array([[16, 12, 15, 11, 12,  1, 10, 16],
        [ 1, 13,  9,  7,  6, 10, 14,  7]]),
 array([16,  3, 16, 11, 15,  0, 10,  0]))

In [ ]:
# Comprimimos o novo cipher text para obter hash(c')
c_check = cipher_txt_check[0].tobytes() + cipher_txt_check[1].tobytes()

c_line = hashlib.sha3_256(c_check).digest()

In [ ]:
# E por final obtemos K do c' vindo do novo cipher text
K_check = hashlib.sha3_256(K_bar_rec + c_line).hexdigest()

K_check

'cbdec1d9fd14d8d34693ec327083904d1b2c6716d1fbb6852994484d97a47cbe'

## Validação

Por final, podemos validar se o cipher text recuperado é igual a gerado anteriormente e se o K gerado após a desencapsulação também é igual.

In [ ]:
# cipher text termo u
cipher_txt[0] == cipher_txt_check[0]

array([[ True,  True,  True,  True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True,  True,  True,  True]])

In [ ]:
# cipher text termo v
cipher_txt[1] == cipher_txt_check[1]

array([ True,  True,  True,  True,  True,  True,  True,  True])

In [ ]:
# shared_key K
K == K_check

True

In [ ]:
# Para termos uma melhor visualização da igualdade dos K's,
# podemos mostrar suas seguintes hashes(K)
K, K_check

('cbdec1d9fd14d8d34693ec327083904d1b2c6716d1fbb6852994484d97a47cbe',
 'cbdec1d9fd14d8d34693ec327083904d1b2c6716d1fbb6852994484d97a47cbe')

No algoritmo real do Kyber temos que se c' for igual a c, então é gerado a chave K, caso contrário ele gera K = ( z || H(c) ), onde z é um segredo armazanedo na chave privada como fallback, evitando assim ataques adaptativos (CCA-secure).

# Conclusão
O CRYSTALS-Kyber baseia sua segurança na dificuldade computacional do problema Module-LWE, para o qual atualmente não são conhecidos algoritmos clássicos nem quânticos de tempo polinomial capazes de resolvê-lo, diferente dos outros algoritmos assimétricos como o RSA que utiliza fatoração de números, e o ECDH que se baseia em curvas elípticas nos quais são facilmente quebradas pelo Shor tendo um hardware quântico potente o suficiente para isso, e isso pode ser daqui a 10 anos, 5 anos, ou até mesmo em 3 anos pelo avanço exponencial da tecnologia nos dias de hoje. Então o que fazer agora? Devemos nos preparar e substituir o RSA e ECDH aos poucos como o método de hibridização com o Kyber, no qual vai ajudar a proteger caso um dos dois algoritmos clássicos falharem.

Além da resistência ao algoritmo de Shor, o Kyber foi projetado para oferecer alta eficiência computacional, chaves relativamente pequenas e excelente desempenho em dispositivos com recursos limitados. Essas características contribuíram para sua seleção pelo NIST como padrão para troca de chaves pós-quântica. Outro aspecto importante é sua segurança CCA, obtida através do mecanismo de reencapsulação e da validação do ciphertext recebido, impedindo diversos ataques adaptativos.

Embora esta implementação utilize parâmetros reduzidos e substitua a aritmética em anéis de polinômios por operações matriciais simplificadas, ela preserva a lógica conceitual do algoritmo original, permitindo compreender cada etapa do processo de geração de chaves, encapsulamento, desencapsulação e validação da chave compartilhada antes de estudar a implementação completa do CRYSTALS-Kyber, no qual eu estarei disponibilizando posteriormente em outro pequeno projeto para podermos conhecer melhor a fundo o Kyber em sua biblioteca real.